# TrDGL-FuzzVn — B3 Q3/Q4 crossed micro-rerun after stop-token repair

This notebook is a **new diagnostic contract**, not an append to the immutable two-seed checkpoint. It tests the root-cause repair on both tuned quantizations under matched prompts and seeds.

Design: 5 representative PyTorch APIs × 2 seeds × 2 quantizations × 2 serving modes (original chat vs repaired `B3Chat`) = 40 generation events. Because only a 16 GB T4 is currently available, this resource-bounded diagnostic uses a 256-token ceiling and partial GPU offload; it is not the frozen 600-token campaign. Every raw output is persisted before interpretation.

The notebook does **not** claim to complete the three missing five-seed shards, candidate novelty triage, the raw Atlas intervention, or certified numerical bounds.

*Co-authored with CoCo*

In [ ]:
import os, sys, subprocess, json, hashlib, time, re, ast, platform
from pathlib import Path
from datetime import datetime, timezone

WORK=Path('/content/trdgl_b3_q3_q4')
MODEL_DIR=WORK/'models'; OUT=WORK/'outputs'
MODEL_DIR.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
print(platform.platform())
subprocess.run(['nvidia-smi'], check=False)


In [ ]:
# Install exact runtime and connector. CUDA build is required for 26B Q4 on an L4/A100 class GPU.
subprocess.check_call([sys.executable,'-m','pip','install','-q','snowflake-connector-python>=3.12','pandas','nbformat'])
subprocess.check_call(['bash','-lc', 'CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install -q --upgrade --force-reinstall --no-cache-dir llama-cpp-python==0.3.23'])
from llama_cpp import Llama
import snowflake.connector


In [ ]:
# Pinned model provenance and stage paths
MODELS={
 'Q3_K_M': {
  'file':'gemma4-26b-a4b-qa.Q3_K_M.gguf',
  'stage':'@AI_LAB.PHASE_A.TRDGL_GGUF_STAGE/models/trdgl_gemma4_26b_a4b_qa/rev_7e25063a3755/gemma4-26b-a4b-qa.Q3_K_M.gguf',
  'size':13286716704,
  'sha256':'2f35a80c395f31c1b39003d1e4e14df6b06796f343bb670c37d53828a96e482b'},
 'Q4_K_M': {
  'file':'gemma4-26b-a4b-qa.Q4_K_M.gguf',
  'stage':'@AI_LAB.PHASE_A.TRDGL_GGUF_STAGE/models/trdgl_gemma4_26b_a4b_qa/rev_7e25063a3755/q4_k_m/gemma4-26b-a4b-qa.Q4_K_M.gguf',
  'size':16795999008,
  'sha256':'ea4a99b6dbdd3fedc9c20838e22dd2fcf272ed0d84360d42deaf0b15e955a068'}
}
SEEDS=[12011,19001]
APIS=['torch.tensor','torch.sum','torch.linalg.solve','torch.sparse.mm','torch.autograd.grad']
DECODING=dict(max_tokens=256,temperature=0.2,top_p=0.95,top_k=40,min_p=0.05,repeat_penalty=1.08)
STOP=['<end_of_turn>','<start_of_turn>','<eos>']
print(json.dumps({'models':MODELS,'seeds':SEEDS,'apis':APIS,'decoding':DECODING},indent=2))


In [ ]:
# Download from Snowflake internal stage and verify the original source bytes.
required=['SNOWFLAKE_ACCOUNT','SNOWFLAKE_USER','SNOWFLAKE_PASSWORD']
missing=[k for k in required if not os.environ.get(k)]
if missing: raise RuntimeError(f'Missing environment variables: {missing}')
conn=snowflake.connector.connect(account=os.environ['SNOWFLAKE_ACCOUNT'],user=os.environ['SNOWFLAKE_USER'],password=os.environ['SNOWFLAKE_PASSWORD'],role=os.environ.get('SNOWFLAKE_ROLE','ACCOUNTADMIN'),warehouse=os.environ.get('SNOWFLAKE_WAREHOUSE','COMPUTE_WH'),database='AI_LAB',schema='PHASE_A')
cur=conn.cursor()
try:
 for q,m in MODELS.items():
  local=MODEL_DIR/m['file']
  if not local.exists():
   cur.execute(f"GET {m['stage']} 'file://{MODEL_DIR.as_posix()}' OVERWRITE=TRUE PARALLEL=16")
  h=hashlib.sha256()
  with local.open('rb') as f:
   for b in iter(lambda:f.read(16*1024*1024),b''): h.update(b)
  got=h.hexdigest(); size=local.stat().st_size
  print(q,size,got)
  assert size==m['size'],(q,size,m['size'])
  assert got==m['sha256'],(q,got,m['sha256'])
finally:
 cur.close(); conn.close()


In [ ]:
# Load the frozen documentation snapshot copied beside this notebook by the runner.
snapshot_path=Path('/content/documentation_snapshot.json')
DOCS=json.loads(snapshot_path.read_text(encoding='utf-8'))
assert all(api in DOCS for api in APIS)

def full_prompt(api,seed):
 doc=DOCS[api]['doc'][:6000]
 return f'''You are generating one standalone Python test for PyTorch API {api}.
Generation seed: {seed}.
Return Python code only, no Markdown fences and no prose.
Requirements:
- import torch;
- deterministically seed torch with {seed};
- call the exact target API {api};
- use bounded small tensors and no network or file access;
- execute the test at top level;
- include at least one meaningful oracle that can fail, such as a shape/dtype/value/gradient relation or eager-versus-alternative agreement;
- do not swallow exceptions and do not use unconditional assertions.
Frozen documentation:
{doc}
'''


In [ ]:
MARKER_RE=re.compile(r'(?i)(?:<)?(?:start_of_turn|end_of_turn|eos)(?:>)?|^\s*(?:model|assistant)\s*\n')

def clean_text(text):
 text=MARKER_RE.sub('',text)
 return text.strip()

def extract_code(text):
 text=clean_text(text)
 m=re.search(r'```(?:python)?\s*(.*?)```',text,re.S|re.I)
 return (m.group(1) if m else text).strip()

def analyze(text,api):
 code=extract_code(text)
 try: ast.parse(code); parseable=True
 except SyntaxError: parseable=False
 target=(api in code)
 oracle=bool(re.search(r'\bassert\b|torch\.testing|allclose|raises|grad',code))
 marker=bool(re.search(r'(?i)start_of_turn|end_of_turn|<eos>',text))
 return {'cleaned':code,'parseable':parseable,'target_call_present':target,'oracle_heuristic':oracle,'marker_leak':marker}

def original_chat(llm,prompt):
 return llm.create_chat_completion(messages=[{'role':'user','content':prompt}],**DECODING)

def repaired_chat(llm,prompt):
 return llm.create_chat_completion(messages=[{'role':'user','content':prompt}],stop=STOP,**DECODING)


In [ ]:
# Crossed order avoids quantization/order confounding within this diagnostic.
events=[]
started=datetime.now(timezone.utc)
orders={12011:['Q3_K_M','Q4_K_M'],19001:['Q4_K_M','Q3_K_M']}
for seed in SEEDS:
 for quant in orders[seed]:
  m=MODELS[quant]; model_path=MODEL_DIR/m['file']
  print('LOADING',quant,model_path)
  llm=Llama(model_path=str(model_path),n_ctx=4096,n_batch=256,n_gpu_layers=32,verbose=False,seed=seed)
  for api in APIS:
   prompt=full_prompt(api,seed)
   prompt_sha=hashlib.sha256(prompt.encode()).hexdigest()
   for mode,fn in [('original',original_chat),('B3Chat_repaired',repaired_chat)]:
    t=time.perf_counter(); resp=fn(llm,prompt); elapsed=time.perf_counter()-t
    choice=resp['choices'][0]; raw=choice['message']['content']; finish=choice.get('finish_reason')
    a=analyze(raw,api)
    row={'schema_version':'trdgl_b3_q3_q4_micro_v1','timestamp_utc':datetime.now(timezone.utc).isoformat(),'seed':seed,'quantization':quant,'mode':mode,'api':api,'prompt_sha256':prompt_sha,'raw_output':raw,'raw_output_sha256':hashlib.sha256(raw.encode()).hexdigest(),'finish_reason':finish,'generation_seconds':elapsed,**a}
    events.append(row)
    with (OUT/'events.jsonl').open('a',encoding='utf-8') as f: f.write(json.dumps(row,ensure_ascii=False)+'\n')
    print(seed,quant,mode,api,finish,a['parseable'],a['marker_leak'],round(elapsed,2))
  del llm
  import gc; gc.collect()


In [ ]:
import pandas as pd
DF=pd.DataFrame(events)
summary=(DF.groupby(['quantization','mode']).agg(events=('api','size'),parseable=('parseable','sum'),target=('target_call_present','sum'),oracle=('oracle_heuristic','sum'),marker_leak=('marker_leak','sum'),mean_seconds=('generation_seconds','mean')).reset_index())
summary['clean_rate']=1-summary['marker_leak']/summary['events']
display(summary)
summary.to_csv(OUT/'summary.csv',index=False)
manifest={'schema_version':'trdgl_b3_q3_q4_micro_manifest_v1','started_utc':started.isoformat(),'completed_utc':datetime.now(timezone.utc).isoformat(),'design':{'seeds':SEEDS,'apis':APIS,'orders':orders,'modes':['original','B3Chat_repaired'],'decoding':DECODING,'stop':STOP},'models':MODELS,'event_count':len(events),'events_sha256':hashlib.sha256((OUT/'events.jsonl').read_bytes()).hexdigest(),'summary':summary.to_dict(orient='records'),'environment':{'python':platform.python_version(),'platform':platform.platform()}}
(OUT/'run_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print(json.dumps(manifest,indent=2,default=str))


## Interpretation boundary

A clean stop repair on Q3 and Q4 supports the narrow conclusion that the earlier B3 collapse was a serving/template-stop mismatch and is not explained by Q3 quantization alone. It does **not** retroactively alter the immutable 960-event checkpoint. Any repaired outputs require a new runner version and run signature.

The next separate experiments remain: complete three frozen seed shards under an explicitly versioned repaired contract, replay the compile candidate in its pinned framework environment, run the Linux compiled numerical matrix, and recover the raw Atlas corpus.